### **Implement Linear Regression using the scikit-learn library or Pytorch and train the model on the processed dataset.**

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Load the Dataset
try:
    df = pd.read_csv('50_Startups.csv')
except FileNotFoundError:
    # Fallback: Generate synthetic 50_Startups data if file is missing
    print("Warning: '50_Startups.csv' not found. Generating synthetic data for demonstration...")
    np.random.seed(42)
    df = pd.DataFrame({
        'R&D Spend': np.random.rand(50) * 100000,
        'Administration': np.random.rand(50) * 100000,
        'Marketing Spend': np.random.rand(50) * 100000,
        'State': np.random.choice(['New York', 'California', 'Florida'], 50),
        'Profit': np.random.rand(50) * 200000
    })

# 2. Preprocessing
# Split Features (X) and Target (y)
X = df.iloc[:, :-1].values  # All columns except 'Profit'
y = df.iloc[:, -1].values.reshape(-1, 1) # 'Profit' column

# Handle Categorical 'State' column (Index 3) & Scale Features
# We use ColumnTransformer to apply OneHotEncoder to 'State' and keep others
ct = ColumnTransformer(transformers=[
    ('encoder', OneHotEncoder(drop='first'), [3])
], remainder='passthrough')

X_processed = ct.fit_transform(X)

# Feature Scaling (Crucial for Gradient Descent convergence in the next step)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X_processed)
y_scaled = scaler_y.fit_transform(y)

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

# 4. Implement Linear Regression (Scikit-Learn)
sklearn_model = LinearRegression()
sklearn_model.fit(X_train, y_train)

# 5. Output Results
print("Scikit-Learn Model Trained Successfully")
print(f"Sklearn Intercept (Bias): {sklearn_model.intercept_[0]:.4f}")
print(f"Sklearn Coefficients (Weights): {sklearn_model.coef_}")

Scikit-Learn Model Trained Successfully
Sklearn Intercept (Bias): -0.0084
Sklearn Coefficients (Weights): [[ 1.09752547e-02  8.29592320e-05  9.17483064e-01 -4.78161240e-02
   9.05824209e-02]]


### **Implement Linear Regression from scratch (without using any ML libraries for the model, create your own function to calculate gradient descent). Compare the learned coefficients and intercept from your custom implementation with those obtained using scikit-learn or Pytorch.** 

In [ ]:
import numpy as np
import pandas as pd

class CustomLinearRegression:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # 1. Initialize parameters (weights and bias)
        self.weights = np.zeros((n_features, 1))
        self.bias = 0

        # 2. Gradient Descent Loop
        for _ in range(self.epochs):
            # Forward pass: Calculate predictions (y = Xw + b)
            y_pred = np.dot(X, self.weights) + self.bias

            # Calculate gradients (derivatives of MSE)
            # dw = (2/n) * X.T * (y_pred - y)
            dw = (2 / n_samples) * np.dot(X.T, (y_pred - y))
            # db = (2/n) * sum(y_pred - y)
            db = (2 / n_samples) * np.sum(y_pred - y)

            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # Optional: Track loss (MSE) for debugging
            loss = np.mean((y_pred - y)**2)
            self.loss_history.append(loss)

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

# Train the Custom Model 
# We use the same scaled data (X_train, y_train) from the previous cell
custom_model = CustomLinearRegression(learning_rate=0.1, epochs=2000)
custom_model.fit(X_train, y_train)

# Compare Results 
# Retrieve Scikit-Learn parameters (from previous cell)
sklearn_w = sklearn_model.coef_.flatten()
sklearn_b = sklearn_model.intercept_[0]

# Retrieve Custom parameters
custom_w = custom_model.weights.flatten()
custom_b = custom_model.bias

# Create Comparison Table
comparison_data = {
    'Parameter': ['Intercept (Bias)'] + [f'Weight_Feat_{i}' for i in range(len(sklearn_w))],
    'Scikit-Learn': [sklearn_b] + list(sklearn_w),
    'Custom GD': [custom_b] + list(custom_w)
}

df_comparison = pd.DataFrame(comparison_data)

# Calculate difference to show precision
df_comparison['Difference'] = df_comparison['Scikit-Learn'] - df_comparison['Custom GD']

print("\nModel Parameter Comparison:")
print(df_comparison.round(6).to_string(index=False))

# Validation Check
if np.allclose(sklearn_w, custom_w, atol=1e-2) and np.isclose(sklearn_b, custom_b, atol=1e-2):
    print("\n SUCCESS: Custom Gradient Descent matches Scikit-Learn logic")
else:
    print("\nNote: Values differ. Try increasing epochs or adjusting learning rate.")


Model Parameter Comparison:
       Parameter  Scikit-Learn  Custom GD  Difference
Intercept (Bias)     -0.008427  -0.008427         0.0
   Weight_Feat_0      0.010975   0.010975        -0.0
   Weight_Feat_1      0.000083   0.000083        -0.0
   Weight_Feat_2      0.917483   0.917483         0.0
   Weight_Feat_3     -0.047816  -0.047816        -0.0
   Weight_Feat_4      0.090582   0.090582        -0.0

 SUCCESS: Custom implementation matches Scikit-Learn logic
